# M4 parity gate — official LeRobot baseline (`select_fruit`)

Locked upstream reference for `lerobot/smolvla_vlabench` on VLABench `select_fruit`.
Mirrors LeRobot docs / CI: 10 episodes, native task instruction, `select_action()` cadence.

Artifacts land in `artifacts/m4_parity/lerobot_baseline.json`.

## 1 — Headless rendering

In [ ]:
import os

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

## 2 — Install VLABench + LeRobot (pinned SHA)

In [ ]:
%%bash
set -euo pipefail

cd /content
VLABENCH_PIN=cf588fe60c0c7282174fe979f5913170cfe69017
RRT_PIN=e51d95ee489a225220d6ae2a764c4111f6ba7d85
EVAL_VENV=/content/m4-baseline-venv

apt-get update -qq
apt-get install -y -qq cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev python3.10-venv

if [ ! -d "$EVAL_VENV" ]; then
  python3.10 -m venv "$EVAL_VENV" || {
    python3.10 -m venv --without-pip "$EVAL_VENV"
    curl -fsSL https://bootstrap.pypa.io/get-pip.py | "$EVAL_VENV/bin/python"
  }
fi

source "$EVAL_VENV/bin/activate"
python -m pip install -U pip
python -m pip install "lerobot[smolvla]>=0.5"

if [ ! -d VLABench/.git ]; then
  git clone https://github.com/OpenMOSS/VLABench VLABench
fi
git -C VLABench checkout "$VLABENCH_PIN"

if [ ! -d rrt-algorithms/.git ]; then
  git clone https://github.com/motion-planning/rrt-algorithms rrt-algorithms
fi
git -C rrt-algorithms checkout "$RRT_PIN"
touch rrt-algorithms/rrt_algorithms/rrt/__init__.py
touch rrt-algorithms/rrt_algorithms/search_space/__init__.py
touch rrt-algorithms/rrt_algorithms/utilities/__init__.py

python -m pip install -e rrt-algorithms -e VLABench
python VLABench/scripts/download_assets.py
python -c "import lerobot, VLABench; print('baseline deps OK', lerobot.__version__)"

## 3 — Run locked `lerobot-eval` (10 episodes)

In [ ]:
%%bash
set -euo pipefail

source /content/m4-baseline-venv/bin/activate
export MUJOCO_GL=egl PYOPENGL_PLATFORM=egl

ARTIFACT_DIR=/content/m4_parity_artifacts
LOG="$ARTIFACT_DIR/lerobot_baseline.log"
mkdir -p "$ARTIFACT_DIR"

lerobot-eval \
  --policy.path=lerobot/smolvla_vlabench \
  --env.type=vlabench \
  --env.task=select_fruit \
  --eval.batch_size=1 \
  --eval.n_episodes=10 \
  --eval.use_async_envs=false \
  --policy.device=cuda \
  '--rename_map={"observation.images.image": "observation.images.camera1", "observation.images.second_image": "observation.images.camera2", "observation.images.wrist_image": "observation.images.camera3"}' \
  2>&1 | tee "$LOG"

echo "log: $LOG"

## 4 — Parse metrics + write artifact JSON

Edit `SUCCESS_COUNT` if the log parser misses the summary line.

In [ ]:
import json
import re
from pathlib import Path

ARTIFACT_DIR = Path('/content/m4_parity_artifacts')
LOG = ARTIFACT_DIR / 'lerobot_baseline.log'
text = LOG.read_text(encoding='utf-8', errors='replace')

# LeRobot eval summaries vary by version; try common patterns.
successes = None
for pattern in (
    r'success(?:es)?[:\s]+(\d+)\s*/\s*(\d+)',
    r'success[_ ]rate[:\s]+([0-9.]+)',
    r'avg[_ ]success[:\s]+([0-9.]+)',
):
    match = re.search(pattern, text, flags=re.IGNORECASE)
    if not match:
        continue
    if len(match.groups()) == 2:
        successes = int(match.group(1))
        break
    rate = float(match.group(1))
    successes = round(rate * 10)
    break

if successes is None:
    raise RuntimeError(
        'Could not parse success count from lerobot-eval log. '
        'Set SUCCESS_COUNT manually below and re-run this cell.'
    )

N_EPISODES = 10
payload = {
    'gate': 'm4_parity',
    'track': 'lerobot_baseline',
    'task': 'select_fruit',
    'policy': 'lerobot/smolvla_vlabench',
    'vlabench_sha': 'cf588fe60c0c7282174fe979f5913170cfe69017',
    'n_episodes': N_EPISODES,
    'successes': successes,
    'success_rate': successes / N_EPISODES,
    'log_path': str(LOG),
}
out = ARTIFACT_DIR / 'lerobot_baseline.json'
out.write_text(json.dumps(payload, indent=2) + '\n', encoding='utf-8')
print(json.dumps(payload, indent=2))
print('artifact:', out)